# 0. Trendlines V4 Geometry Research Lab

This notebook is a hands-on view of the complete frozen Trendlines V4 research family. It uses the fixed `pivot_window=3` and `history=300` semantics, closed native-timeframe candles, and the canonical `geometry.v2` view. Pathfinding shows structural/current_valid/secondary; pivot consensus shows structural(span_first)/local(consensus_first). Each chart and table is descriptive; the notebook does not make a directional decision.

# 1. Environment / imports

In [ ]:
import pandas as pd
from IPython.display import display

from libs.models.trendlines_v4.core_v2 import HISTORY_CAPACITY_BARS, PIVOT_WINDOW
from libs.models.trendlines_v4.research_lab import (
    TVLC_CDN_URL,
    analyze_frames,
    build_causal_replay_payload,
    compare_asset_frames,
    fetch_native_window_async,
    geometry_rows,
    pivot_consensus_rows,
    pivot_rows,
    render_pivot_consensus_chart,
    render_causal_scrolling_replay,
    render_tvlc_chart,
    role_transition_rows,
    snapshot_json,
    snapshot_summary_rows,
)

print("Trendlines V4 research lab")
print({"pivot_window": PIVOT_WINDOW, "history_capacity_bars": HISTORY_CAPACITY_BARS})
print({"lightweight_charts": TVLC_CDN_URL})
print(
    "Pinned CDN: https://unpkg.com/lightweight-charts@5.2.1/dist/lightweight-charts.standalone.production.js"
)

# 2. User Controls

In [ ]:
ASSET = "BTCUSDT"
TIMEFRAMES = ("1h", "4h")
PRIMARY_TIMEFRAME = "1h"
START_AT = "2026-06-01T00:00:00Z"
END_AT = "2026-07-01T00:00:00Z"
ALLOW_PROVIDER_FETCH = False

VIEW_BARS = None  # None = full fetched date window; positive int = trailing N candles
SCROLL_START_OFFSET = 120
SCROLL_STEP_SIZE = 6
SCROLL_STEPS = 30

COMPARE_ASSETS = ("BTCUSDT", "ETHUSDT", "SOLUSDT")
COMPARE_TIMEFRAME = "4h"

print("Fixed model: pivot_window=3, history=300")
print("Model history is always 300 bars. VIEW_BARS changes display history only.")
print(
    {"asset": ASSET, "timeframes": TIMEFRAMES, "provider_fetch": ALLOW_PROVIDER_FETCH}
)

# 3. Data Loading

Set `INJECTED_FRAMES` to a mapping of native-timeframe DataFrames for offline work. Provider access is opt-in and uses close time as the V4 cutoff.

In [ ]:
INJECTED_FRAMES = {}

if ALLOW_PROVIDER_FETCH:
    mtf_data = {}
    for tf in TIMEFRAMES:
        mtf_data[tf] = await fetch_native_window_async(ASSET, tf, START_AT, END_AT)
else:
    mtf_data = {tf: INJECTED_FRAMES[tf] for tf in TIMEFRAMES if tf in INJECTED_FRAMES}
    if not mtf_data:
        print(
            "Provider fetch is disabled. Inject native frames or set ALLOW_PROVIDER_FETCH=True."
        )

In [ ]:
inventory = []
for tf, frame in mtf_data.items():
    inventory.append(
        {
            "timeframe": tf,
            "rows": len(frame),
            "first_closed_at": (
                frame["closed_at"] if "closed_at" in frame else frame["close_time"]
            ).iloc[0]
            if len(frame)
            else None,
            "last_closed_at": (
                frame["closed_at"] if "closed_at" in frame else frame["close_time"]
            ).iloc[-1]
            if len(frame)
            else None,
            "source_mode": "provider" if ALLOW_PROVIDER_FETCH else "injected",
        }
    )
display(pd.DataFrame(inventory))

# 4. Run V4 Geometry

Each native timeframe is analyzed independently by the canonical V2 engine.

In [ ]:
results = analyze_frames(mtf_data) if mtf_data else {}
print(f"Analyzed {len(results)} native timeframe(s).")

# 5. Snapshot Summary Table

In [ ]:
summary = snapshot_summary_rows(results) if results else ()
display(pd.DataFrame(summary))

# 6. Interactive TVLC Charts — Pathfinding

The helper emits notebook-native HTML in this cell for pathfinding structural, current_valid, and secondary roles. Lightweight Charts is pinned to v5.2.1; the chart data ends at each snapshot cutoff.

In [ ]:
for tf in TIMEFRAMES:
    if tf in mtf_data:
        render_tvlc_chart(
            mtf_data[tf],
            results[tf],
            timeframe=tf,
            view_bars=VIEW_BARS,
        )

# 7. Interactive TVLC Charts — Pivot Consensus

Pivot-consensus structural (`span_first`) and local (`consensus_first`) selections are rendered separately over the same candles and cutoff. Their selection uses only the fixed trailing 300-bar model history; `VIEW_BARS` changes display only.

In [ ]:
for tf in TIMEFRAMES:
    if tf in mtf_data:
        render_pivot_consensus_chart(
            mtf_data[tf],
            results[tf],
            timeframe=tf,
            title=f"{tf} · Pivot consensus · span_first / consensus_first",
            view_bars=VIEW_BARS,
        )

# 8. Geometry Diagnostics

In [ ]:
for tf in TIMEFRAMES:
    if tf in mtf_data:
        display(pd.DataFrame(geometry_rows(mtf_data[tf], results[tf], timeframe=tf)))

# 9. Pivot Diagnostics

In [ ]:
for tf in TIMEFRAMES:
    if tf in mtf_data:
        display(pd.DataFrame(pivot_rows(mtf_data[tf], timeframe=tf)))

# 10. Pivot-Consensus Diagnostics

Selected candidate facts are descriptive and come directly from the frozen F1B selectors; no new score or threshold is introduced.

In [ ]:
for tf in TIMEFRAMES:
    if tf in mtf_data:
        display(
            pd.DataFrame(
                pivot_consensus_rows(mtf_data[tf], results[tf], timeframe=tf)
            )
        )

# 11. Causal Scrolling Replay

The viewer precomputes prefix-only snapshots and displays Prev, Next, Play, Stop, and Slider controls in one notebook output cell.

In [ ]:
if PRIMARY_TIMEFRAME in mtf_data:
    replay_steps = build_causal_replay_payload(
        mtf_data[PRIMARY_TIMEFRAME],
        timeframe=PRIMARY_TIMEFRAME,
        start_offset=SCROLL_START_OFFSET,
        step_size=SCROLL_STEP_SIZE,
        steps=SCROLL_STEPS,
    )
    render_causal_scrolling_replay(
        mtf_data[PRIMARY_TIMEFRAME],
        timeframe=PRIMARY_TIMEFRAME,
        start_offset=SCROLL_START_OFFSET,
        step_size=SCROLL_STEP_SIZE,
        steps=SCROLL_STEPS,
    )

# 12. Role Transition Diagnostics

In [ ]:
if PRIMARY_TIMEFRAME in mtf_data:
    transitions = role_transition_rows(
        mtf_data[PRIMARY_TIMEFRAME],
        timeframe=PRIMARY_TIMEFRAME,
        start_offset=SCROLL_START_OFFSET,
        step_size=SCROLL_STEP_SIZE,
        steps=SCROLL_STEPS,
    )
    display(pd.DataFrame(transitions))

# 13. Independent Multi-Timeframe Context

The 1h and latest closed 4h views are placed side-by-side as independent context.

In [ ]:
context_rows = []
for tf in ("1h", "4h"):
    if tf in results:
        context_rows.extend(snapshot_summary_rows(results[tf], timeframe=tf))
display(pd.DataFrame(context_rows))
print("MTF_CONTEXT_INCONCLUSIVE — independent views only.")

# 14. Multi-Asset Comparison

Supply native frames in `COMPARE_FRAMES` to inspect factual role availability and geometry context across assets at one native timeframe.

In [ ]:
COMPARE_FRAMES = {}
if COMPARE_FRAMES:
    display(
        pd.DataFrame(compare_asset_frames(COMPARE_FRAMES, timeframe=COMPARE_TIMEFRAME))
    )
else:
    print("Multi-asset comparison is waiting for injected native frames.")

# 15. Export Current Snapshot

The payload below is JSON-serializable and contains factual V2 geometry fields.

In [ ]:
snapshot_export = {tf: snapshot_json(snapshot) for tf, snapshot in results.items()}
for tf, payload in snapshot_export.items():
    print(f"--- {tf} ---")
    print(payload)